# 03 · Reranker로 결정 추천 만들기

**목표**: 사례와 정책 후보를 rerank하여 `approve`, `manual-review`, `decline` 중 하나를 **추천**합니다. reranker는 새 판단 규칙을 학습하는 모델이 아니라, 입력 사례와 후보 정책의 관련도를 점수화하는 모델입니다.

이 실습은 실제 승인·반려를 실행하지 않습니다. 점수 차이가 작거나 근거가 약하면 보수적으로 `manual-review`를 반환하고 담당자 검토를 요구합니다. 실행 전 루트 `.env`에 `LLM_MODEL`과 배포된 `RERANK_MODEL`을 설정하세요. LiteLLM/Cohere 호환 endpoint를 쓸 때만 `RERANK_ENDPOINT`, `RERANK_API_KEY`를 추가합니다.

<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #1976d2; background: #eef6ff; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
  <div style="font-size: .82em; font-weight: 700; color: #1976d2; letter-spacing: .02em;">📘  개념 · 실습 단계</div>
  <div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">1. 정책 기반 추천 만들기 (UI)</h2>
<p style="margin: 7px 0;">고객 행동을 입력하고 <b>추천 생성</b>을 누르세요. 결과는 정책 후보의 순위, score, margin, 사람 검토 필요 여부를 함께 보여줍니다.</p>
<ol style="margin: 7px 0 0; padding-left: 1.3em;"><li>고객 행동을 입력합니다.</li><li>필요하면 ⚙️ 설정에서 정책 문서, 최소 margin, 최소 score를 조정합니다.</li><li>추천 결과의 <code>requires_human_review</code>가 true이면 자동 승인·반려 대신 담당자 검토로 보냅니다.</li></ol>
<p style="margin: 7px 0;">UI는 추천과 근거 조회만 수행합니다. 승인·반려·검토 큐 등록 같은 외부 업무 상태는 변경하지 않습니다.</p>
  </div>
</div>

In [1]:
from pathlib import Path
import sys

project_root = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / "1_gen_ai_hub" / "src").exists())
for module_path in (project_root / "1_gen_ai_hub" / "src", project_root / "2_gen_ai_hub_llm"):
    if str(module_path) not in sys.path:
        sys.path.insert(0, str(module_path))

from src.notebook_ui import build_decision_reranker_ui
from src.reranker.lab_core import (
    APPLICATION_DECISIONS, DEFAULT_MIN_MARGIN, DEFAULT_MIN_SCORE,
    DEFAULT_ORDER_POLICY_DOCUMENT, run_decision_recommendation,
)

candidate_options = [
    {"id": candidate.id, "label": candidate.label, "description": candidate.description}
    for candidate in APPLICATION_DECISIONS
]
decision_ui = build_decision_reranker_ui(
    run_decision_recommendation,
    default_policy_document=DEFAULT_ORDER_POLICY_DOCUMENT,
    candidates=candidate_options,
    default_min_margin=DEFAULT_MIN_MARGIN,
    default_min_score=DEFAULT_MIN_SCORE,
)
decision_ui.launch(inline=True, share=False, show_error=True, server_name="127.0.0.1", width="100%", height=900)

* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.


<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #1a73e8; background: #e8f0fe; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
  <div style="font-size: .82em; font-weight: 700; color: #1a73e8; letter-spacing: .02em;">✅  체크포인트</div>
  <div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">체크포인트</h2>
<ul style="margin: 7px 0 0; padding-left: 1.3em;"><li>추천 결과에 정책 후보 순위와 <code>requires_human_review</code>가 보인다.</li><li>점수나 margin이 약한 결과는 <code>manual-review</code>로 반환된다.</li><li>UI가 추천만 생성하며 실제 업무 상태를 변경하지 않는다는 것을 확인한다.</li></ul>
<p style="margin: 7px 0;">문제가 없으면 다음으로 진행하세요. 아래 코드는 같은 코어 API를 Python으로 살펴보는 선택 심화 자료입니다.</p>
  </div>
</div>

<hr style="border: 0; border-top: 2px solid #c7d2e1; margin: 32px 0 14px;">
<div style="margin: 0 0 18px; color: #4b5f78; font-size: .86em; font-weight: 700; letter-spacing: .04em;">🧩 아래부터 코드 베이스 실습</div>

<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #5f6368; background: #f1f3f4; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
  <div style="font-size: .82em; font-weight: 700; color: #5f6368; letter-spacing: .02em;">🔎  선택 심화</div>
  <div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">선택: 테스트된 코어 API로 추천 근거 읽기</h2>
<p style="margin: 7px 0;">아래 셀은 UI와 같은 <code>run_decision_recommendation()</code>을 호출합니다. LLM 정책 요약, reranker 호출, score·margin 검증, 사람 검토 게이트는 코어에 있으며 노트북에서 다시 구현하지 않습니다.</p>
  </div>
</div>

In [2]:
from pathlib import Path
import sys

project_root = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / "1_gen_ai_hub" / "src").exists())
for module_path in (project_root / "1_gen_ai_hub" / "src", project_root / "2_gen_ai_hub_llm"):
    if str(module_path) not in sys.path:
        sys.path.insert(0, str(module_path))

from lab_common.config import load_settings
from src.reranker.lab_core import run_decision_recommendation

settings = load_settings()

<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #1976d2; background: #eef6ff; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
  <div style="font-size: .82em; font-weight: 700; color: #1976d2; letter-spacing: .02em;">📘  개념 · 실습 단계</div>
  <div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">1. 사례를 정책 후보와 비교하기</h2>
<p style="margin: 7px 0;"><code>ranked_candidates</code>는 후보별 관련도 순위이고, <code>margin</code>은 1위와 2위의 점수 차이입니다. 원격 semantic reranker의 최소 score·margin은 운영 전에 라벨된 사례로 평가·조정해야 합니다.</p>
  </div>
</div>

In [3]:
case = "인증된 고객이 주문 간격 60초 이상을 지키며 1분에 1회 주문을 시도했다. 중복, 실패, 한도 초과는 없다."
recommendation = run_decision_recommendation(case, settings=settings)
recommendation

{'case': '인증된 고객이 주문 간격 60초 이상을 지키며 1분에 1회 주문을 시도했다. 중복, 실패, 한도 초과는 없다.',
 'recommendation': 'manual-review',
 'recommendation_label': '모니터링 필요',
 'requires_human_review': True,
 'margin': 0.02863673,
 'engine': 'llm_policy_btp_cohere_rerank',
 'reason': 'LLM이 정책 문서와 행동 사실을 요약한 뒤 Cohere가 후보 관련도를 순위화했습니다. 1·2위 점수 차이 0.029가 최소 margin 0.120보다 작습니다.',
 'policy_summary': '인증된 고객이 주문 간격 60초 이상, 1분당 1회 주문을 시도하여 주문 빈도 정상 범위에 해당한다. 중복 주문, 반복 실패, 한도 초과는 확인되지 않았다.',
 'policy_id': 'sha256:8d3b5a18bd2d',
 'decision_thresholds': {'min_score': 0.45, 'min_margin': 0.12},
 'ranked_candidates': [{'rank': 1,
   'id': 'approve',
   'label': '정상 고객',
   'score': 0.9389352},
  {'rank': 2, 'id': 'manual-review', 'label': '모니터링 필요', 'score': 0.91029847},
  {'rank': 3, 'id': 'decline', 'label': '고위험 고객', 'score': 0.84004915}],
 'action': '추천만 생성했습니다. 승인·반려 상태는 변경하지 않았습니다.'}

<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #1976d2; background: #eef6ff; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
  <div style="font-size: .82em; font-weight: 700; color: #1976d2; letter-spacing: .02em;">📘  개념 · 실습 단계</div>
  <div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">2. 반환된 근거와 검토 게이트 읽기</h2>
<p style="margin: 7px 0;">코어가 반환한 recommendation, score, margin, 사람 검토 필요 여부를 그대로 확인합니다. <code>requires_human_review=True</code>이면 이후 워크플로에서 자동 상태 변경을 막아야 합니다.</p>
  </div>
</div>

In [4]:
print(f"추천: {recommendation['recommendation_label']} ({recommendation['recommendation']})")
print(f"사람 검토 필요: {recommendation['requires_human_review']}")
print(f"margin: {recommendation['margin']:.3f}")
print(f"정책 식별자: {recommendation['policy_id']}")
print(f"reranker 입력 요약: {recommendation['policy_summary']}")
print(f"적용 기준: {recommendation['decision_thresholds']}")
for candidate in recommendation["ranked_candidates"]:
    print(f"{candidate['rank']}. {candidate['label']}: {candidate['score']:.3f}")
print(recommendation["action"])

추천: 모니터링 필요 (manual-review)
사람 검토 필요: True
margin: 0.029
정책 식별자: sha256:8d3b5a18bd2d
reranker 입력 요약: 인증된 고객이 주문 간격 60초 이상, 1분당 1회 주문을 시도하여 주문 빈도 정상 범위에 해당한다. 중복 주문, 반복 실패, 한도 초과는 확인되지 않았다.
적용 기준: {'min_score': 0.45, 'min_margin': 0.12}
1. 정상 고객: 0.939
2. 모니터링 필요: 0.910
3. 고위험 고객: 0.840
추천만 생성했습니다. 승인·반려 상태는 변경하지 않았습니다.


<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #5f6368; background: #f1f3f4; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
  <div style="font-size: .82em; font-weight: 700; color: #5f6368; letter-spacing: .02em;">🔎  선택 심화</div>
  <div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">3. 세 위험 유형을 함께 비교하기</h2>
<p style="margin: 7px 0;">정상·불확실·고위험 사례를 같은 정책과 기준으로 비교합니다. 결과가 무엇이든 <code>requires_human_review=True</code>이면 이후 워크플로에서 자동 상태 변경을 해서는 안 됩니다.</p>
  </div>
</div>

In [5]:
case_matrix = {
    '정상': '인증된 고객이 주문 간격 60초 이상을 지키며 1분에 1회 주문을 시도했다. 중복, 실패, 한도 초과는 없다.',
    '불확실': '필수 서류가 누락되어 신원 정보와 제출 정보가 불일치한다. 추가 검토가 필요하다.',
    '고위험': '같은 배송지에서 여러 계정이 30초 안에 반복 주문했다. 결제수단을 바꾸며 인증 실패가 5회 발생했고, 구매 한도를 피하려는 분할 주문 정황이 있다.',
}
case_recommendations = {}
for case_type, case_text in case_matrix.items():
    outcome = run_decision_recommendation(case_text, settings=settings)
    case_recommendations[case_type] = outcome
    print(f'{case_type}: {outcome["recommendation"]} | score={outcome["ranked_candidates"][0]["score"]:.3f} | margin={outcome["margin"]:.3f} | review={outcome["requires_human_review"]}')

정상: manual-review | score=0.946 | margin=0.029 | review=True
불확실: manual-review | score=0.758 | margin=0.141 | review=True
고위험: manual-review | score=0.934 | margin=0.001 | review=True


<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #1a73e8; background: #e8f0fe; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
  <div style="font-size: .82em; font-weight: 700; color: #1a73e8; letter-spacing: .02em;">✅  체크포인트</div>
  <div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">중간 체크포인트</h2>
<ol style="margin: 7px 0 0; padding-left: 1.3em;"><li>추천 결과의 정책 식별자, reranker 입력 요약, score·margin·검토 기준을 확인했다.</li><li>정상·불확실·고위험 사례의 결과를 비교했다.</li><li>score는 확률이나 사실 검증 결과가 아니며, 낮거나 경합하는 결과는 검토로 전환해야 함을 확인했다.</li></ol>
  </div>
</div>

<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #1976d2; background: #eef6ff; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
  <div style="font-size: .82em; font-weight: 700; color: #1976d2; letter-spacing: .02em;">📘  실습 4</div>
  <div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">같은 사례에서 검토 게이트 조정하기</h2>
<p style="margin: 7px 0;">최소 score와 margin은 모델이 정하는 값이 아니라 운영자가 라벨된 사례를 바탕으로 정하는 안전 기준입니다. 같은 경계 사례에 완화·보수 기준을 적용하고 자동 추천 범위가 어떻게 달라지는지 확인하세요.</p>
  </div>
</div>


In [6]:
borderline_case = '인증된 고객이 1분 동안 두 번 주문을 시도했다. 한 번의 결제 실패가 있었지만 중복 주문과 한도 초과는 확인되지 않았다.'
gate_configs = [
    {'name': '완화', 'min_score': 0.30, 'min_margin': 0.05},
    {'name': '기본', 'min_score': 0.45, 'min_margin': 0.12},
    {'name': '보수', 'min_score': 0.65, 'min_margin': 0.25},
]
for config in gate_configs:
    outcome = run_decision_recommendation(
        borderline_case, settings=settings, min_score=config['min_score'], min_margin=config['min_margin']
    )
    print(f'{config["name"]}: {outcome["recommendation"]} | review={outcome["requires_human_review"]} | {outcome["reason"]}')

완화: manual-review | review=True | LLM이 정책 문서와 행동 사실을 요약한 뒤 Cohere가 후보 관련도를 순위화했습니다. 가장 관련도 높은 후보가 manual-review입니다.
기본: manual-review | review=True | LLM이 정책 문서와 행동 사실을 요약한 뒤 Cohere가 후보 관련도를 순위화했습니다. 가장 관련도 높은 후보가 manual-review입니다. 1·2위 점수 차이 0.058가 최소 margin 0.120보다 작습니다.
보수: manual-review | review=True | LLM이 정책 문서와 행동 사실을 요약한 뒤 Cohere가 후보 관련도를 순위화했습니다. 가장 관련도 높은 후보가 manual-review입니다. 1·2위 점수 차이 0.052가 최소 margin 0.250보다 작습니다.


<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #1976d2; background: #eef6ff; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
  <div style="font-size: .82em; font-weight: 700; color: #1976d2; letter-spacing: .02em;">📘  실습 5</div>
  <div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">작은 라벨 평가셋으로 점검하기</h2>
<p style="margin: 7px 0;">앞의 세 사례는 설명용 미니 평가셋입니다. 기대 정책 결과와 실제 추천을 비교하고, 검토로 전환된 비율을 확인하세요. 운영에서는 조직이 검토한 과거 사례를 더 많이 사용해야 합니다.</p>
  </div>
</div>


In [7]:
expected_decisions = {'정상': 'approve', '불확실': 'manual-review', '고위험': 'decline'}
matches = 0
review_count = 0
for case_type, expected in expected_decisions.items():
    outcome = case_recommendations[case_type]
    actual = outcome['recommendation']
    matches += actual == expected
    review_count += outcome['requires_human_review']
    print(f'{case_type}: 기대={expected}, 추천={actual}, 검토={outcome["requires_human_review"]}')
print()
print(f'기대 결과 일치: {matches}/{len(expected_decisions)}')
print(f'사람 검토 전환: {review_count}/{len(expected_decisions)}')
print('주의: reranker score는 확률이 아니며, 이 수치는 설명용 사례에서만 계산한 값입니다.')

정상: 기대=approve, 추천=manual-review, 검토=True
불확실: 기대=manual-review, 추천=manual-review, 검토=True
고위험: 기대=decline, 추천=manual-review, 검토=True

기대 결과 일치: 1/3
사람 검토 전환: 3/3
주의: reranker score는 확률이 아니며, 이 수치는 설명용 사례에서만 계산한 값입니다.


<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #1a73e8; background: #e8f0fe; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
  <div style="font-size: .82em; font-weight: 700; color: #1a73e8; letter-spacing: .02em;">✅ 미션</div>
  <div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">정책 triage 평가 설계</h2>
<p style="margin: 7px 0;">사례를 6개 이상 추가하고 기대 정책 결과를 담당자 검토 기준으로 작성하세요. 두 가지 이상의 threshold 조합에 대해 결과 일치율과 사람 검토 전환율을 비교합니다. 이 모듈의 출력은 추천·근거·검토 전환까지만 담당하며 승인·반려 같은 상태 변경은 수행하지 않습니다.</p>
  </div>
</div>


In [8]:
my_labeled_cases = [
    # {'case_type': 'TODO', 'case_text': '정책 사실만 포함한 사례', 'expected': 'approve'},
]
my_gate_configs = [
    # {'name': '기본', 'min_score': 0.45, 'min_margin': 0.12},
]

# 작성 후: 각 기준별 추천 일치율, manual-review 전환율, 잘못된 자동 추천을 기록하세요.
# 실제 승인·반려 상태를 변경하는 코드는 이 실습에 추가하지 않습니다.